#  TEMPORAL ROBUSTNESS OF FRAUD-DETECTION MODELS: DATASET SHIFT, CALIBRATION AND EXPLANATION STABILITY ON REAL AND SIMULATED TRANSACTION DATA


In [ ]:
%pip install -q kagglehub xgboost

In [ ]:
import os, glob, json, random, warnings, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, rankdata
from scipy.special import logit

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, roc_curve
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
random.seed(0); np.random.seed(0)

# ================== CONFIG ==================
QUICK = False                      # True = tiny smoke test (do NOT use for paper results)
SEEDS = [0] if QUICK else [0, 1, 2]
N_RANDOM_SPLITS = 2 if QUICK else 5
N_BLOCKS = 5                       # rolling chronological test blocks
TRAIN_FRAC = 0.50                  # first 50% of time = training period
VAL_FRAC = 0.15                    # last 15% (in time) of training period = calibration set
N_BOOT = 100 if QUICK else 300
N_BOOT_SLOPE = 50 if QUICK else 100
IMP_NEG = 8000                     # negatives per evaluation set for permutation importance
IMP_REPEATS = 2 if QUICK else 3
SPARKOV_MAX_ROWS = 20000 if QUICK else 300000   # uniform random subsample keeps prevalence and time order
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

MAIN = ["LogReg", "RandomForest", "HistGB", "XGBoost", "MLP"]
ABL = ["LogReg (unweighted)", "HistGB (unweighted)", "XGBoost (unweighted)"]

## 1. Core helpers: splits, models, metrics, calibration

In [ ]:
def get_models(seed, pos_w):
    return {
        "LogReg": make_pipeline(StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)),
        "RandomForest": RandomForestClassifier(n_estimators=150, min_samples_leaf=2, max_depth=14,
            class_weight="balanced_subsample", n_jobs=-1, random_state=seed),
        "HistGB": HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200,
            class_weight="balanced", random_state=seed),
        "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, scale_pos_weight=pos_w, tree_method="hist", eval_metric="aucpr",
            n_jobs=-1, random_state=seed),
        "MLP": make_pipeline(StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=200, random_state=seed)),
        # ---- ablations: same learners WITHOUT class weighting (RQ3) ----
        "LogReg (unweighted)": make_pipeline(StandardScaler(),
            LogisticRegression(max_iter=2000, random_state=seed)),
        "HistGB (unweighted)": HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200, random_state=seed),
        "XGBoost (unweighted)": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, scale_pos_weight=1, tree_method="hist", eval_metric="aucpr",
            n_jobs=-1, random_state=seed),
    }

def temporal_blocks(n):
    i_tr = int(TRAIN_FRAC * n); i_fit = int(i_tr * (1 - VAL_FRAC))
    step = int(round((1 - TRAIN_FRAC) * n / N_BLOCKS))
    edges = [(i_tr + k * step, (i_tr + (k + 1) * step) if k < N_BLOCKS - 1 else n) for k in range(N_BLOCKS)]
    return i_fit, i_tr, edges

def random_split(y, seed):
    """Stratified random split with the SAME train size; test set cut into equal-size random blocks."""
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, train_size=TRAIN_FRAC, stratify=y, random_state=seed)
    fit, val = train_test_split(tr, test_size=VAL_FRAC, stratify=y[tr], random_state=seed)
    te = np.random.default_rng(seed).permutation(te)
    return fit, val, np.array_split(te, N_BLOCKS)

def ece_score(y, p, n_bins=15):
    o = np.argsort(p); y, p = y[o], p[o]
    bins = np.array_split(np.arange(len(p)), n_bins)
    return float(sum(len(b) / len(p) * abs(y[b].mean() - p[b].mean()) for b in bins))

def recall_at_fpr(y, s, target=0.01):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.interp(target, fpr, tpr))

def all_metrics(y, p):
    return dict(AP=average_precision_score(y, p), AUC=roc_auc_score(y, p),
                Brier=brier_score_loss(y, p), ECE=ece_score(y, p), R_at_1FPR=recall_at_fpr(y, p))

def platt_fit(s_val, y_val, eps=1e-6):
    z = logit(np.clip(s_val, eps, 1 - eps)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(z, y_val)
    return lambda s: lr.predict_proba(logit(np.clip(s, eps, 1 - eps)).reshape(-1, 1))[:, 1]

def kendall_w(rank_matrix):
    """rows = raters (time blocks), cols = items (models). 1 = identical rankings in every block."""
    m, n = rank_matrix.shape
    R = rank_matrix.sum(0)
    return float(12 * ((R - R.mean()) ** 2).sum() / (m ** 2 * (n ** 3 - n)))

def topk_jaccard(a, b, k=5):
    A, B = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k])
    return len(A & B) / len(A | B)

## 2. Experiment runner


In [ ]:
def run_study(tag, X, y, feats):
    n = len(y); rows = []; keep = {}; imp = {}
    i_fit, i_tr, tb = temporal_blocks(n)
    fit_idx, val_idx = np.arange(i_fit), np.arange(i_fit, i_tr)
    blocks_idx = [np.arange(a, b) for a, b in tb]

    rng = np.random.default_rng(0)
    def sub(idx):
        pos, neg = idx[y[idx] == 1], idx[y[idx] == 0]
        return np.r_[pos, rng.choice(neg, min(IMP_NEG, len(neg)), replace=False)]
    imp_sets = {"val": sub(val_idx),
                "early": sub(np.r_[blocks_idx[0], blocks_idx[1]]),
                "late": sub(np.r_[blocks_idx[-2], blocks_idx[-1]])}

    def one_split(split, rep, fit_i, val_i, blocks, seed, keep_scores, do_imp):
        yf = y[fit_i]; pos_w = (yf == 0).sum() / max(yf.sum(), 1)
        for name, model in get_models(seed, pos_w).items():
            model.fit(X[fit_i], yf)
            cal = platt_fit(model.predict_proba(X[val_i])[:, 1], y[val_i])
            for k, bi in enumerate(blocks, 1):
                s = model.predict_proba(X[bi])[:, 1]; sc = cal(s); yt = y[bi]
                m, mc = all_metrics(yt, s), all_metrics(yt, sc)
                rows.append(dict(split=split, rep=rep, block=k, model=name, seed=seed, **m,
                                 Brier_cal=mc["Brier"], ECE_cal=mc["ECE"], mean_pred=s.mean(),
                                 mean_pred_cal=sc.mean(), obs=yt.mean(), n_pos=int(yt.sum())))
                if keep_scores: keep[(name, k)] = (yt, s, sc)
            if do_imp and name in MAIN:
                for sn, idx in imp_sets.items():
                    r = permutation_importance(model, X[idx], y[idx], scoring="average_precision",
                                               n_repeats=IMP_REPEATS, random_state=0, n_jobs=1)
                    imp[(name, seed, sn)] = r.importances_mean
            del model

    for seed in SEEDS:
        one_split("temporal", seed, fit_idx, val_idx, blocks_idx, seed, seed == SEEDS[0], True)
        print(f"[{tag}] temporal seed {seed} done")
    for r in range(N_RANDOM_SPLITS):
        f, v, b = random_split(y, r)
        one_split("random", r, f, v, b, r, False, False)
        print(f"[{tag}] random split {r} done")
    res = pd.DataFrame(rows)
    res.to_csv(f"{OUT}/{tag}_results_raw.csv", index=False)
    return dict(tag=tag, res=res, keep=keep, imp=imp, feats=feats, n=n, prev=float(y.mean()),
                idx=(fit_idx, val_idx, blocks_idx))

## 3. Load both datasets with the Kaggle Hub API


In [ ]:
import kagglehub
# kagglehub.login()   # uncomment only if downloads ask for credentials

def haversine(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

def load_creditcard():
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    d = pd.read_csv(glob.glob(os.path.join(path, "**", "creditcard.csv"), recursive=True)[0])
    d = d.sort_values("Time").reset_index(drop=True)
    d["LogAmount"] = np.log1p(d["Amount"])
    feats = [f"V{i}" for i in range(1, 29)] + ["LogAmount"]      # Time excluded on purpose
    return d[feats].values.astype(np.float32), d["Class"].values.astype(int), feats, d["Time"].values

def load_sparkov():
    path = kagglehub.dataset_download("kartik2112/fraud-detection")
    WANT = {"trans_date_trans_time", "category", "amt", "gender", "lat", "long", "city_pop",
            "dob", "unix_time", "merch_lat", "merch_long", "is_fraud"}
    files = sorted(glob.glob(os.path.join(path, "**", "fraud*.csv"), recursive=True))
    d = pd.concat([pd.read_csv(f, usecols=lambda c: c in WANT) for f in files], ignore_index=True)
    d = d.sort_values("unix_time").reset_index(drop=True)
    if len(d) > SPARKOV_MAX_ROWS:   # uniform subsample: preserves prevalence AND chronological order
        d = d.sample(SPARKOV_MAX_ROWS, random_state=0).sort_values("unix_time").reset_index(drop=True)
    ts, dob = pd.to_datetime(d["trans_date_trans_time"]), pd.to_datetime(d["dob"])
    hour = ts.dt.hour + ts.dt.minute / 60
    f = pd.DataFrame({
        "log_amt": np.log1p(d["amt"]), "log_city_pop": np.log1p(d["city_pop"]),
        "dist_km": haversine(d["lat"].values, d["long"].values, d["merch_lat"].values, d["merch_long"].values),
        "age": (ts - dob).dt.days / 365.25,
        "hour_sin": np.sin(2 * np.pi * hour / 24), "hour_cos": np.cos(2 * np.pi * hour / 24),
        "dow": ts.dt.dayofweek, "is_male": (d["gender"] == "M").astype(int)})
    f = pd.concat([f, pd.get_dummies(d["category"], prefix="cat").astype(int)], axis=1)
    return f.values.astype(np.float32), d["is_fraud"].values.astype(int), list(f.columns), d["unix_time"].values

DATA = {}
for tag, loader in [("creditcard", load_creditcard), ("sparkov", load_sparkov)]:
    X, y, feats, t = loader()
    DATA[tag] = (X, y, feats, t)
    print(f"{tag}: rows={len(y):,} features={len(feats)} frauds={int(y.sum()):,} ({y.mean()*100:.3f}%)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, (tag, (X, y, feats, t)) in zip(axes, DATA.items()):
    dec = np.floor(np.arange(len(y)) / len(y) * 10).astype(int)
    ax.bar(range(10), pd.Series(y).groupby(dec).mean() * 100)
    ax.set_title(tag); ax.set_xlabel("Chronological decile"); ax.set_ylabel("Fraud rate (%)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig0_prevalence.png"); plt.show()

## 4. Run the experiments (the slow step)

In [ ]:
STUDY = {}
for tag, (X, y, feats, t) in DATA.items():
    STUDY[tag] = run_study(tag, X, y, feats)
print("All studies finished.")

## 5. RQ1 + RQ2: optimism gap, decay slope, rank stability across time blocks

In [ ]:
def gap_ci(S, name, seed=0):
    r = S["res"]
    rr = r[(r.split == "random") & (r.model == name)].groupby("rep").AP.mean().values
    tt = r[(r.split == "temporal") & (r.model == name)].groupby("block").AP.mean().values
    rng = np.random.default_rng(seed)
    d = [rng.choice(rr, len(rr)).mean() - rng.choice(tt, len(tt)).mean() for _ in range(N_BOOT)]
    return rr.mean() - tt.mean(), *np.percentile(d, [2.5, 97.5])

def slope_boot(S, name, seed=0):
    r = S["res"]; t = r[(r.split == "temporal") & (r.model == name)].groupby("block").AP.mean()
    slope = np.polyfit(t.index.values, t.values, 1)[0]
    rng = np.random.default_rng(seed); reps = []
    for _ in range(N_BOOT_SLOPE):
        aps = []
        for k in range(1, N_BLOCKS + 1):
            y, s, _ = S["keep"][(name, k)]; idx = rng.integers(0, len(y), len(y))
            aps.append(average_precision_score(y[idx], s[idx]) if y[idx].sum() > 0 else np.nan)
        if not np.isnan(aps).any(): reps.append(np.polyfit(np.arange(1, N_BLOCKS + 1), aps, 1)[0])
    return slope, *np.percentile(reps, [2.5, 97.5])

def performance_table(S):
    r = S["res"]; rows = []
    for name in MAIN:
        t = r[(r.split == "temporal") & (r.model == name)]
        tb = t.groupby("block").AP.mean()
        g, glo, ghi = gap_ci(S, name); sl, slo, shi = slope_boot(S, name)
        rows.append(dict(model=name,
            AP_random=r[(r.split == "random") & (r.model == name)].AP.mean(),
            AP_temporal=tb.mean(), AP_block_sd=tb.std(ddof=1), AP_worst_block=tb.min(), AP_best_block=tb.max(),
            gap=g, gap_lo=glo, gap_hi=ghi, slope_per_block=sl, slope_lo=slo, slope_hi=shi,
            AUC_temporal=t.AUC.mean(), R1FPR_temporal=t.R_at_1FPR.mean()))
    return pd.DataFrame(rows).set_index("model").round(4)

def rank_stats(S):
    t = S["res"]; t = t[(t.split == "temporal") & (t.model.isin(MAIN))]
    apm = t.groupby(["model", "block"]).AP.mean().unstack().loc[MAIN]
    ranks = np.vstack([rankdata(-apm[k].values) for k in apm.columns])
    winners = [apm.index[int(np.argmin(ranks[i]))] for i in range(ranks.shape[0])]
    return apm, winners, kendall_w(ranks)

PERF, RANK = {}, {}
for tag, S in STUDY.items():
    print(f"\n===== {tag} =====")
    P = performance_table(S); PERF[tag] = P; P.to_csv(f"{OUT}/{tag}_performance.csv"); display(P)
    apm, winners, W = rank_stats(S); RANK[tag] = dict(apm=apm, winners=winners, W=W)
    apm.round(3).to_csv(f"{OUT}/{tag}_ap_by_block.csv"); display(apm.round(3))
    print("Winner per block:", winners, f"| Kendall W across blocks = {W:.2f} (1 = identical ranking every block)")

    fig, ax = plt.subplots(figsize=(7, 3.8))
    for i, m in enumerate(MAIN):
        c = f"C{i}"; ax.plot(range(1, N_BLOCKS + 1), apm.loc[m].values, "o-", color=c, label=m)
        ax.axhline(P.loc[m, "AP_random"], color=c, ls="--", lw=.8, alpha=.7)
    ax.set_xlabel("Chronological test block (no retraining)"); ax.set_ylabel("Average precision")
    ax.set_title(f"{tag}: temporal AP per block (dashed = random-split AP)"); ax.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.savefig(f"{OUT}/{tag}_fig1_ap_blocks.png"); plt.show()

### Statistical comparison on the pooled test period (paired bootstrap)

In [ ]:
def pooled(S, name):
    ys = np.concatenate([S["keep"][(name, k)][0] for k in range(1, N_BLOCKS + 1)])
    ss = np.concatenate([S["keep"][(name, k)][1] for k in range(1, N_BLOCKS + 1)])
    return ys, ss

PAIR = {}
for tag, S in STUDY.items():
    sc = {};
    for n_ in MAIN: y_, sc[n_] = pooled(S, n_)
    rng = np.random.default_rng(0); draws = {n_: [] for n_ in MAIN}
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(y_), len(y_))
        if y_[idx].sum() == 0: continue
        for n_ in MAIN: draws[n_].append(average_precision_score(y_[idx], sc[n_][idx]))
    draws = {k: np.array(v) for k, v in draws.items()}
    best = PERF[tag]["AP_temporal"].idxmax(); rows = []
    for n_ in MAIN:
        lo, med, hi = np.percentile(draws[n_], [2.5, 50, 97.5])
        row = dict(model=n_, pooled_AP_median=med, CI_low=lo, CI_high=hi)
        if n_ != best:
            d = draws[best] - draws[n_]; dlo, dhi = np.percentile(d, [2.5, 97.5])
            row.update(diff_vs_best=d.mean(), diff_lo=dlo, diff_hi=dhi, best_significantly_better=bool(dlo > 0))
        rows.append(row)
    T = pd.DataFrame(rows).set_index("model").round(4); PAIR[tag] = dict(table=T, best=best)
    T.to_csv(f"{OUT}/{tag}_pooled_ap_and_paired_tests.csv")
    print(f"\n===== {tag} | best by mean block AP: {best} | diff_vs_best = best - model =====")
    display(T)

## 6. Drift diagnostics (with a control)


In [ ]:
def adv_auc(A, B, n=15000, seed=0):
    rng = np.random.default_rng(seed)
    A = A[rng.choice(len(A), min(n, len(A)), replace=False)]
    B = B[rng.choice(len(B), min(n, len(B)), replace=False)]
    Xa = np.vstack([A, B]); ya = np.r_[np.zeros(len(A)), np.ones(len(B))]
    Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=.3, stratify=ya, random_state=seed)
    clf = HistGradientBoostingClassifier(max_iter=100, random_state=seed).fit(Xtr, ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

def psi_vec(A, B, bins=10):
    out = []
    for j in range(A.shape[1]):
        a, b = A[:, j], B[:, j]
        e = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
        if len(e) < 3: out.append(np.nan); continue
        e[0], e[-1] = -np.inf, np.inf
        pa = np.clip(np.histogram(a, e)[0] / len(a), 1e-4, None)
        pb = np.clip(np.histogram(b, e)[0] / len(b), 1e-4, None)
        out.append(float(np.sum((pb - pa) * np.log(pb / pa))))
    return np.array(out)

DRIFT = {}
for tag, S in STUDY.items():
    X = DATA[tag][0]; fit_idx, val_idx, bl = S["idx"]
    rng = np.random.default_rng(0); perm = rng.permutation(fit_idx); h = len(perm) // 2
    rows = [dict(comparison="CONTROL: random halves of training period",
                 adv_AUC=adv_auc(X[perm[:h]], X[perm[h:]]), mean_PSI=np.nanmean(psi_vec(X[perm[:h]], X[perm[h:]])))]
    rows.append(dict(comparison="train vs validation (adjacent in time)", adv_AUC=adv_auc(X[fit_idx], X[val_idx]),
                     mean_PSI=np.nanmean(psi_vec(X[fit_idx], X[val_idx]))))
    for k, b in enumerate(bl, 1):
        rows.append(dict(comparison=f"train vs test block {k}", adv_AUC=adv_auc(X[fit_idx], X[b]),
                         mean_PSI=np.nanmean(psi_vec(X[fit_idx], X[b]))))
    D = pd.DataFrame(rows).set_index("comparison").round(4); DRIFT[tag] = D
    D.to_csv(f"{OUT}/{tag}_drift.csv"); print(f"\n===== {tag} ====="); display(D)
    pv = psi_vec(X[fit_idx], X[bl[-1]]); top = np.argsort(-np.nan_to_num(pv))[:3]
    print("Most shifted features in last block (PSI):", [(S["feats"][j], round(float(pv[j]), 3)) for j in top])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for tag, D in DRIFT.items():
    blk = D.loc[[i for i in D.index if i.startswith("train vs test block")]]
    axes[0].plot(range(1, N_BLOCKS + 1), blk["adv_AUC"].values, "o-", label=tag)
    axes[1].plot(range(1, N_BLOCKS + 1), blk["mean_PSI"].values, "o-", label=tag)
axes[0].axhline(.5, color="k", ls=":"); axes[0].set_title("Adversarial AUC vs training"); axes[1].set_title("Mean PSI vs training")
for a in axes: a.set_xlabel("Test block"); a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_drift.png"); plt.show()

## 7. RQ3 - Calibration, risk tiers and the class-weighting ablation
`CITL` = calibration-in-the-large = mean predicted probability / observed fraud rate (1 is perfect).

In [ ]:
TIERS = [(0, .005), (.005, .02), (.02, .10), (.10, 1.0)]

def tier_table(S, names):
    rows = []
    for name in names:
        y = np.concatenate([S["keep"][(name, k)][0] for k in range(1, N_BLOCKS + 1)])
        s = np.concatenate([S["keep"][(name, k)][1] for k in range(1, N_BLOCKS + 1)])
        sc = np.concatenate([S["keep"][(name, k)][2] for k in range(1, N_BLOCKS + 1)])
        o = np.argsort(-s); n = len(y)
        for lo, hi in TIERS:
            ii = o[int(lo * n):int(hi * n)]
            rows.append(dict(model=name, tier=f"top {lo*100:g}-{hi*100:g}%", n=len(ii), n_pos=int(y[ii].sum()),
                             observed=y[ii].mean(), pred_raw=s[ii].mean(), pred_cal=sc[ii].mean()))
    return pd.DataFrame(rows)

CAL, TIER = {}, {}
for tag, S in STUDY.items():
    t = S["res"]; t = t[t.split == "temporal"]
    C = t.groupby("model").agg(Brier=("Brier", "mean"), Brier_cal=("Brier_cal", "mean"), ECE=("ECE", "mean"),
                               ECE_cal=("ECE_cal", "mean"), mp=("mean_pred", "mean"), mpc=("mean_pred_cal", "mean"),
                               obs=("obs", "mean"))
    C["CITL_raw"] = C.mp / C.obs; C["CITL_cal"] = C.mpc / C.obs
    C = C.drop(columns=["mp", "mpc", "obs"]).loc[MAIN + ABL]
    CAL[tag] = C; C.round(5).to_csv(f"{OUT}/{tag}_calibration.csv")
    print(f"\n===== {tag} ====="); display(C.round(5))
    T = tier_table(S, MAIN + ABL); TIER[tag] = T; T.to_csv(f"{OUT}/{tag}_risk_tier_calibration.csv")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    for ax, col, title in zip(axes, ["pred_raw", "pred_cal"], ["Raw scores", "After Platt scaling"]):
        lim = [T[["observed", col]].replace(0, np.nan).min().min() * .5, 1]
        ax.plot(lim, lim, "k--", lw=1)
        for i, m in enumerate(MAIN + ABL):
            g = T[(T.model == m) & (T.observed > 0)]
            ax.plot(g[col], g["observed"], "o-" if m in MAIN else "s:", ms=4, color=f"C{i}", label=m)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Mean predicted probability"); ax.set_title(title)
    axes[0].set_ylabel("Observed fraud rate"); axes[0].legend(fontsize=6)
    plt.suptitle(f"{tag}: risk-tier calibration (tiers by score percentile, pooled test period)")
    plt.tight_layout(); plt.savefig(f"{OUT}/{tag}_fig2_risk_tier_calibration.png"); plt.show()

## 8. RQ4 - Explanation stability (permutation importance across time)


In [ ]:
def stability_table(S):
    rows = []
    for name in MAIN:
        a = []
        for sd in SEEDS:
            v, e, l = (S["imp"][(name, sd, k)] for k in ["val", "early", "late"])
            a.append((spearmanr(v, l)[0], spearmanr(e, l)[0], topk_jaccard(v, l)))
        a = np.array(a)
        rows.append(dict(model=name, rho_val_late=np.nanmean(a[:, 0]), rho_val_late_sd=np.nanstd(a[:, 0]),
                         rho_early_late=np.nanmean(a[:, 1]), top5_jaccard_val_late=np.nanmean(a[:, 2])))
    return pd.DataFrame(rows).set_index("model").round(3)

STAB = {}
fig, axes = plt.subplots(1, len(STUDY), figsize=(4.5 * len(STUDY), 3.3), sharey=True)
for ax, (tag, S) in zip(np.atleast_1d(axes), STUDY.items()):
    St = stability_table(S); STAB[tag] = St; St.to_csv(f"{OUT}/{tag}_explanation_stability.csv")
    print(f"\n===== {tag} ====="); display(St)
    ax.bar(MAIN, St["rho_val_late"], yerr=St["rho_val_late_sd"], capsize=3)
    ax.set_title(tag); ax.set_ylabel("Spearman: importance(val) vs importance(late)"); ax.tick_params(axis="x", rotation=40)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_stability.png"); plt.show()

## 9. RQ5 - Cross-dataset consistency

In [ ]:
rows = []
for tag in STUDY:
    for m in MAIN:
        rows.append(dict(dataset=tag, model=m, AP_temporal=PERF[tag].loc[m, "AP_temporal"],
                         gap=PERF[tag].loc[m, "gap"], slope_per_block=PERF[tag].loc[m, "slope_per_block"],
                         rho_val_late=STAB[tag].loc[m, "rho_val_late"],
                         ECE_raw=CAL[tag].loc[m, "ECE"], ECE_cal=CAL[tag].loc[m, "ECE_cal"]))
CROSS = pd.DataFrame(rows); CROSS.round(4).to_csv(f"{OUT}/cross_dataset_summary.csv", index=False)
display(CROSS.pivot(index="model", columns="dataset", values="AP_temporal").loc[MAIN].round(3))

tags = list(STUDY)
if len(tags) == 2:
    a = CROSS[CROSS.dataset == tags[0]].set_index("model").loc[MAIN, "AP_temporal"]
    b = CROSS[CROSS.dataset == tags[1]].set_index("model").loc[MAIN, "AP_temporal"]
    r_, p_ = spearmanr(a, b); print(f"Model-ranking agreement across datasets: Spearman = {r_:.2f} (p = {p_:.2f}, n = 5)")

r1 = spearmanr(CROSS.rho_val_late, CROSS.AP_temporal)
r2 = spearmanr(CROSS.rho_val_late, CROSS.slope_per_block)
print(f"Pooled (dataset x model, n={len(CROSS)}; points are not independent):")
print(f"  stability vs temporal AP : rho={r1[0]:.2f}, p={r1[1]:.2f}")
print(f"  stability vs AP slope    : rho={r2[0]:.2f}, p={r2[1]:.2f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
for tag, mk in zip(tags, ["o", "s"]):
    g = CROSS[CROSS.dataset == tag]
    ax.scatter(g.rho_val_late, g.AP_temporal, marker=mk, label=tag)
    for _, r in g.iterrows(): ax.annotate(r.model, (r.rho_val_late, r.AP_temporal), fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Explanation stability (Spearman val vs late)"); ax.set_ylabel("Mean temporal AP"); ax.legend()
ax.set_title("Accuracy vs explanation stability")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_accuracy_vs_stability.png"); plt.show()

## 10. Auto-generated findings (verify against the tables before using any sentence)

In [ ]:
for tag in STUDY:
    P, C, D, R = PERF[tag], CAL[tag], DRIFT[tag], RANK[tag]
    print(f"\n### {tag}  (rows={STUDY[tag]['n']:,}, fraud rate={STUDY[tag]['prev']*100:.3f}%)")
    print(f"- Best mean temporal AP: {P.AP_temporal.idxmax()} ({P.AP_temporal.max():.3f}); worst: {P.AP_temporal.idxmin()} ({P.AP_temporal.min():.3f}).")
    sig_gap = P.index[P.gap_lo > 0].tolist()
    print(f"- RQ1: optimism gap (random - temporal AP) ranges {P.gap.min():.3f} to {P.gap.max():.3f}; 95% CI excludes 0 for {len(sig_gap)}/5 models {sig_gap}.")
    dec = P.index[P.slope_hi < 0].tolist(); inc = P.index[P.slope_lo > 0].tolist()
    print(f"- RQ2: significant AP decline over blocks for {dec}; significant increase for {inc}; the rest are indistinguishable from flat.")
    print(f"- RQ2: block winners = {R['winners']}; Kendall W = {R['W']:.2f}.")
    ctrl = D.iloc[0]["adv_AUC"]; blk = D.iloc[2:]["adv_AUC"]
    print(f"- Drift: control adversarial AUC = {ctrl:.3f}; train-vs-block AUC ranges {blk.min():.3f}-{blk.max():.3f}.")
    for base in ["LogReg", "HistGB", "XGBoost"]:
        print(f"- RQ3 ablation {base}: ECE {C.loc[base,'ECE']:.4f} (weighted) vs {C.loc[base+' (unweighted)','ECE']:.4f} (unweighted); "
              f"after Platt {C.loc[base,'ECE_cal']:.4f} / {C.loc[base+' (unweighted)','ECE_cal']:.4f}.")
    St = STAB[tag]
    print(f"- RQ4: least/most stable explanations: {St.rho_val_late.idxmin()} ({St.rho_val_late.min():.2f}) / {St.rho_val_late.idxmax()} ({St.rho_val_late.max():.2f}).")

In [ ]:
env = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
           sklearn=__import__("sklearn").__version__, xgboost=__import__("xgboost").__version__,
           platform=platform.platform(), seeds=SEEDS, n_random_splits=N_RANDOM_SPLITS, n_blocks=N_BLOCKS,
           n_boot=N_BOOT, quick_mode=QUICK, sparkov_max_rows=SPARKOV_MAX_ROWS)
json.dump(env, open(f"{OUT}/environment.json", "w"), indent=2); print(env)
if QUICK: print("\nWARNING: QUICK mode - results are for smoke-testing only.")

try:
    from google.colab import files
    import shutil
    shutil.make_archive("paper_outputs_v2", "zip", OUT); files.download("paper_outputs_v2.zip")
except ImportError:
    pass